# Phase 5: Customer Churn Prediction

In this phase, we build a machine learning model to predict
customer churn based on behavioral and transactional features.

Objectives:
- Define churn using purchase inactivity
- Validate and clean modeling data
- Handle missing values appropriately
- Train and evaluate classification models
- Identify key drivers of churn

In [1]:
# ============================================
# 1. Import Libraries
# ============================================

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import classification_report

In [ ]:
# ============================================
# 2. Load Feature Dataset
# ============================================

customer_df = pd.read_csv("../data/processed/customer_features.csv")

customer_df.head()

In [ ]:
# ============================================
# 3. Missing Value Analysis
# ============================================

missing_summary = customer_df.isnull().sum()

missing_summary[missing_summary > 0] # all missing columns 

## Define Churn Label

Since explicit churn labels are unavailable, we define churn
based on customer inactivity.

A customer is considered churned if their last purchase
occurred more than 90 days ago.

In [ ]:
# ============================================
# 4. Create Churn Target
# ============================================

churn_threshold = 300

customer_df['churn'] = np.where(
    customer_df['recency_days'] > churn_threshold,
    1,
    0
)

customer_df['churn'].value_counts(normalize=True)

In [ ]:
plt.figure()

customer_df['churn'].value_counts().plot(kind='bar')

plt.title("Churn Distribution")
plt.xlabel("Churn (0 = Active, 1 = Churned)")
plt.ylabel("Number of Customers")

plt.show()

## Feature Selection

We select customer behavioral metrics
as predictors for churn modeling.

In [6]:
# ============================================
# 6. Feature Selection
# ============================================

features = [
    'recency_days',
    'frequency',
    'monetary',
    'avg_order_value',
    'avg_review_score'
]

X = customer_df[features]

y = customer_df['churn']

In [7]:
# ============================================
# 7. Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y

)

In [8]:
# ============================================
# 8. Feature Scaling
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model 1: Logistic Regression

Logistic Regression serves as a baseline model
for churn classification.

In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train_scaled)
X_test_imputed = imputer.transform(X_test_scaled)

In [10]:
# ============================================
# 9. Train Logistic Regression (Baseline Model)
# ============================================
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_model.fit(X_train_imputed, y_train)
y_pred_log = log_model.predict(X_test_imputed)

In [ ]:
cm = confusion_matrix(y_test, y_pred_log)

print("Confusion Matrix:\n", cm)

In [ ]:
RocCurveDisplay.from_estimator(
    log_model,
    X_test_imputed,
    y_test
)

plt.title("Logistic Regression ROC Curve")
plt.show()

## Model 2: Random Forest

Random Forest captures nonlinear relationships
and interactions between features.

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [ ]:
y_prob_rf = rf_model.predict_proba(X_test)[:,1]
print(y_prob_rf)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

# Churn Modeling Insights

Key findings:

- Recency is the strongest predictor of churn.
- Customers with low purchase frequency are
  more likely to become inactive.
- Higher spending customers tend to remain engaged.

Business Recommendations:

- Monitor customers with increasing recency.
- Trigger retention campaigns early.
- Provide incentives for repeat purchases.

## Business Actions

Companies can use this model to:

identify customers likely to churn

send retention offers

create loyalty programs

re-engage inactive customers

In [ ]:
# ============================================
# 11. Save Final Model
# ============================================

import pickle
import os

os.makedirs("../models", exist_ok=True)

pickle.dump(rf_model, open("../models/churn_model.pkl", "wb"))

print("Model is saved successfully.")